# เทรน LoRA ลายผ้าไหมมัดหมี่อีสาน

โน้ตบุ๊กนี้เทรน LoRA บน Stable Diffusion XL ด้วย dataset ที่เตรียมจาก `preprocess.py`

**ก่อนเริ่ม** ไปที่ Runtime → Change runtime type → เลือก **T4 GPU**

| ขั้น | เวลาโดยประมาณบน T4 |
|---|---|
| ติดตั้ง | 3-5 นาที |
| ขยายภาพด้วย Real-ESRGAN | 2-3 นาที |
| เทรน ~860 steps | 45-70 นาที |
| เจนภาพทดสอบทุก checkpoint | 10-15 นาที |

**ข้อควรรู้เรื่องข้อมูล** ภาพต้นฉบับมาจาก PDF ความละเอียดจริงแค่ 422-745px
tile จึงอยู่ที่ 288px เท่านั้น โน้ตบุ๊กนี้ขยายด้วย Real-ESRGAN เป็น 576px ก่อนเทรน
ซึ่งให้ขอบลายคมกว่าการขยายธรรมดามาก แต่ไม่ได้สร้างรายละเอียดที่ต้นฉบับไม่มี
ถ้าถ่ายผ้าจริงที่ 2000px+ ได้เมื่อไร คุณภาพจะกระโดดกว่าการจูนค่าใด ๆ ในนี้


## 1. ตรวจ GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'ยังไม่ได้เปิด GPU — Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 2. ติดตั้ง


In [ ]:
# Colab ติดตั้ง torchao 0.10.0 มาให้ ซึ่ง peft ทุกรุ่นที่ปล่อยแล้วยัง raise ใส่
# (รุ่นบน main แก้เป็นแค่เตือนแล้ว แต่ยังไม่ออกเป็นรุ่นจริง)
# เราไม่ได้ใช้ torchao จึงถอนออก peft จะได้ข้ามไปเอง
!pip uninstall -y -q torchao

# ติดตั้ง diffusers จากซอร์ส สคริปต์ตัวอย่างกับไลบรารีจะได้เป็นรุ่นเดียวกันเสมอ
!git clone -q --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers_src

# --no-deps สำคัญมาก ห้ามให้ pip แตะ torch ของ Colab
# ถ้าปล่อยให้อัปเกรดตามใจ มันจะลง torch รุ่น CPU ทับ แล้ว GPU หายไปทั้งรันไทม์
!pip install -q --no-deps /content/diffusers_src
!pip install -q --no-deps -U peft

import importlib.util
import accelerate, diffusers, peft, torch, transformers

print('torchao ถูกถอนแล้ว:', importlib.util.find_spec('torchao') is None)
print('torch', torch.__version__, '| CUDA ใช้ได้:', torch.cuda.is_available())
print('diffusers', diffusers.__version__, '· peft', peft.__version__)
print('transformers', transformers.__version__, '· accelerate', accelerate.__version__)

assert torch.cuda.is_available(), (
    'torch ไม่มี CUDA แล้ว แปลว่ามีอะไรไปเปลี่ยน torch เป็นรุ่น CPU '
    'ให้ไปที่ Runtime > Disconnect and delete runtime แล้วเริ่มรันใหม่ตั้งแต่เซลล์ 1'
)
print('ติดตั้งเสร็จ')

## 3. อัปโหลด dataset

รัน `python training/make_bundle.py` บนเครื่องก่อน แล้วอัปโหลด `v2_bundle.zip` ที่ได้


In [ ]:
from google.colab import files
import zipfile, pathlib, json

uploaded = files.upload()
name = next(iter(uploaded))

root = pathlib.Path('/content/dataset')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as bundle:
    bundle.extractall(root)

summary = json.loads((root / 'dataset.json').read_text(encoding='utf-8'))
TRIGGER = summary['trigger']
print(summary['tiles'], 'tiles ·', summary['tile_size'], 'px · trigger =', TRIGGER)
print('ตระกูลลาย:', summary['families'])


## 4. ขยายภาพด้วย Real-ESRGAN

tile 288px เล็กเกินกว่าที่ SDXL จะเรียนได้ดี การขยายแบบ bicubic ทำให้ขอบลายฟุ้ง
แล้วโมเดลจะเรียนความเบลอติดไปด้วย Real-ESRGAN รักษาขอบได้ดีกว่ามาก

ถ้าเซลล์นี้พัง ข้ามไปได้ มันจะใช้ภาพขนาดเดิมแทนโดยอัตโนมัติ


In [ ]:
import subprocess, sys, pathlib, shutil
from PIL import Image

SRC = pathlib.Path('/content/dataset/images')
DST = pathlib.Path('/content/dataset_up/images')
DST.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'realesrgan', 'basicsr'], check=True)
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    import cv2

    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    upsampler = RealESRGANer(
        scale=4,
        model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        model=model, tile=0, half=True,
    )
    for path in sorted(SRC.glob('*.png')):
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        output, _ = upsampler.enhance(image, outscale=2)
        cv2.imwrite(str(DST / path.name), output)
    print('ขยายด้วย Real-ESRGAN เรียบร้อย')
except Exception as error:
    print('ขยายไม่สำเร็จ ใช้ภาพขนาดเดิมแทน:', str(error)[:200])
    for path in sorted(SRC.glob('*.png')):
        shutil.copy(path, DST / path.name)

shutil.copy(SRC / 'metadata.jsonl', DST / 'metadata.jsonl')

# augment.py ต้องอ่าน dataset.json คู่กับโฟลเดอร์ images
shutil.copy('/content/dataset/dataset.json', '/content/dataset_up/dataset.json')
print('ขนาด tile ที่จะเทรน:', Image.open(next(DST.glob('*.png'))).size)


## 4.5 เพิ่มข้อมูลด้วยการหมุนเฉดสี

ลายมัดหมี่ลายเดียวกันถูกทอด้วยโทนสีต่างกันจริงในแต่ละชุมชน
การหมุนเฉดจึงสร้างตัวอย่างที่ถูกต้อง ไม่ใช่การปลอมข้อมูล

ทำบน Colab เพราะชุดที่ขยายแล้วใหญ่ราว 400 MB อัปโหลดไม่ไหว
ตรงนี้ขยายภาพก่อนแล้วค่อยหมุนสี จึงขยายแค่ 143 ใบ ไม่ใช่ 2000 ใบ


In [ ]:
!python /content/dataset/augment.py \
  --dataset /content/dataset_up \
  --out /content/dataset_aug \
  --hues 0,45,90,140,190,240,290

import pathlib
TRAIN_DIR = '/content/dataset_aug/images'
print()
print('พร้อมเทรนจาก', TRAIN_DIR,
      len(list(pathlib.Path(TRAIN_DIR).glob('*.png'))), 'tiles')

# diffusers อ่าน caption จาก metadata.jsonl ส่วนไฟล์ .txt เป็นรูปแบบของ kohya
# ลบออกเพื่อไม่ให้ imagefolder ไปหยิบไฟล์ที่ไม่ใช่ภาพเข้ามา
removed = 0
for junk in pathlib.Path(TRAIN_DIR).glob('*.txt'):
    junk.unlink()
    removed += 1
print('ลบไฟล์ .txt ที่ไม่ได้ใช้ออก', removed, 'ไฟล์')
print('เหลือ', len(list(pathlib.Path(TRAIN_DIR).glob('*.png'))), 'ภาพ + metadata.jsonl')


## 5. ตั้งค่าการเทรน

ค่าเหล่านี้ตั้งมาสำหรับ dataset ~143 tiles
ถ้าขยาย dataset เป็น 500+ ให้ลด `EPOCHS` และเพิ่ม `RANK`


In [ ]:
RESOLUTION = 768      # SDXL ชอบ 1024 แต่ข้อมูลเราไม่ถึง 768 คือจุดสมดุล
RANK = 16             # ความจุของ LoRA — 16 พอสำหรับข้อมูลระดับนี้
LR = 1e-4
BATCH = 1             # T4 มี 16GB เท่านั้น
GRAD_ACCUM = 4        # effective batch = 4
EPOCHS = 3             # ข้อมูลเยอะขึ้น 14 เท่า จึงใช้ epoch น้อยลง

import pathlib, math
tiles = len(list(pathlib.Path('/content/dataset_aug/images').glob('*.png')))
steps_per_epoch = math.ceil(tiles / (BATCH * GRAD_ACCUM))
MAX_STEPS = steps_per_epoch * EPOCHS
CHECKPOINT_EVERY = max(150, steps_per_epoch // 2)

print(tiles, 'tiles ·', steps_per_epoch, 'steps/epoch · รวม', MAX_STEPS, 'steps')
print('เก็บ checkpoint ทุก', CHECKPOINT_EVERY, 'steps → ได้', MAX_STEPS // CHECKPOINT_EVERY, 'จุด')
print('ประมาณเวลาบน T4:', round(MAX_STEPS * 2.5 / 60), 'นาที')


## 6. เทรน


In [ ]:
!accelerate launch --mixed_precision=fp16 \
  /content/diffusers_src/examples/text_to_image/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --pretrained_vae_model_name_or_path='madebyollin/sdxl-vae-fp16-fix' \
  --train_data_dir='/content/dataset_aug/images' \
  --caption_column='text' \
  --resolution={RESOLUTION} --random_flip \
  --train_batch_size={BATCH} \
  --gradient_accumulation_steps={GRAD_ACCUM} \
  --gradient_checkpointing \
  --max_train_steps={MAX_STEPS} \
  --checkpointing_steps={CHECKPOINT_EVERY} \
  --learning_rate={LR} \
  --lr_scheduler='cosine' --lr_warmup_steps=50 \
  --rank={RANK} \
  --seed=1337 \
  --mixed_precision='fp16' \
  --output_dir='/content/lora_out'


**ถ้าเจอ CUDA out of memory** ลดทีละขั้น

1. ตั้ง `RESOLUTION = 640` แล้วรันเซลล์ตั้งค่ากับเซลล์เทรนใหม่
2. `RESOLUTION = 512`
3. ยังไม่พอ ให้ไปใช้ SD 1.5 ที่เซลล์ท้ายโน้ตบุ๊ก

> ไม่ได้ใช้ `--use_8bit_adam` แล้ว เพราะ LoRA มีพารามิเตอร์ที่ต้องเทรนน้อยมาก
> สถานะของ optimizer จึงเล็กอยู่แล้ว ไม่คุ้มกับความเสี่ยงที่ bitsandbytes จะติดตั้งไม่ผ่าน


## 7. เจนภาพทดสอบจากทุก checkpoint

ใช้ prompt ชุดเดิมและ seed ชุดเดิมทุก checkpoint เพื่อให้เทียบกันอย่างยุติธรรม

prompt แบ่งสามกลุ่ม กลุ่มแรกทดสอบว่าเรียนสำเร็จไหม กลุ่มสองทดสอบว่าผสมลายใหม่ได้ไหม
ซึ่งเป็นเป้าหมายหลัก ส่วนกลุ่มสามเป็นลายที่ไม่มีใน dataset ใช้จับ overfitting


### ตรวจก่อนว่าการเทรนได้ผลอะไรมาบ้าง

ถ้าเซลล์นี้บอกว่าไม่พบอะไรเลย แปลว่าการเทรนยังไม่สำเร็จ
ให้เลื่อนขึ้นไปดู error ของเซลล์เทรนก่อน อย่าเพิ่งรันเซลล์ถัดไป


In [ ]:
import pathlib

lora_root = pathlib.Path('/content/lora_out')
print('มีโฟลเดอร์ lora_out ไหม:', lora_root.exists())

if lora_root.exists():
    for item in sorted(lora_root.iterdir()):
        weights = list(item.glob('*.safetensors')) if item.is_dir() else []
        mark = ' ← ใช้ได้' if weights else ''
        print(' ', item.name, mark)
    top = list(lora_root.glob('*.safetensors'))
    if top:
        print('  ไฟล์ LoRA ตัวสุดท้าย:', [w.name for w in top], '← ใช้ได้')
else:
    print('ยังไม่มีผลการเทรน ให้กลับไปรันเซลล์ข้อ 6 ให้ผ่านก่อน')


In [ ]:
PROMPTS = {
  'A': TRIGGER + ', naga motif with flame crest, burnt orange ground, cream figures',
  'B': TRIGGER + ', tiered offering motif with finial, violet ground, white figures',
  'C': TRIGGER + ', concentric lozenge lattice with floral centre, indigo ground',
  'D': TRIGGER + ', S-hook motif between warp bands, dark maroon ground, cream figures',
  'E': TRIGGER + ', naga motif inside lozenge lattice, indigo ground, three tones of blue',
  'F': TRIGGER + ', tiered offering motif with hook filler, green ground, gold figures',
  'G': TRIGGER + ', spiral and scroll motif, two tone gold and black',
  'H': TRIGGER + ', lotus motif, pink ground, white figures',
  'I': TRIGGER + ', horizontal banded repeat, teal ground',
  'J': TRIGGER,
}
SEEDS = [1337, 2024, 7777, 42242]
print(len(PROMPTS), 'prompts ×', len(SEEDS), 'seeds =', len(PROMPTS) * len(SEEDS), 'ภาพต่อ checkpoint')


In [ ]:
import torch
assert torch.cuda.is_available(), (
    'ไม่พบ GPU ตอนจะเจนภาพ อย่าเพิ่งโหลดโมเดล 7 GB '
    'ให้ Runtime > Disconnect and delete runtime แล้วเริ่มใหม่ตั้งแต่เซลล์ 1'
)

import torch, pathlib, gc
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
).to('cuda')
pipe.set_progress_bar_config(disable=True)

NEGATIVE = 'photograph of a person, text, watermark, blurry, frame, border'
out_root = pathlib.Path('/content/samples')

out_root.mkdir(parents=True, exist_ok=True)

lora_root = pathlib.Path('/content/lora_out')
if not lora_root.exists():
    raise SystemExit(
        'ไม่พบ /content/lora_out แปลว่าเซลล์เทรน (ข้อ 6) ยังไม่ได้รัน '
        'หรือรันแล้วไม่สำเร็จ ให้เลื่อนขึ้นไปดู error ของเซลล์นั้นก่อน'
    )

checkpoints = sorted(
    lora_root.glob('checkpoint-*'),
    key=lambda p: int(p.name.split('-')[1]),
)
checkpoints.append(lora_root)

usable = [c for c in checkpoints if list(c.glob('*.safetensors'))]
print('พบ checkpoint ที่ใช้ได้', len(usable), 'ตัว:', [c.name for c in usable])
if not usable:
    raise SystemExit(
        'ไม่พบไฟล์ .safetensors เลย แปลว่าการเทรนยังไม่ถึงจุดเก็บ checkpoint แรก '
        '(ตั้งไว้ที่ 250 steps) หรือเทรนล้มกลางทาง ให้รันเซลล์เทรนใหม่'
    )

for ckpt in usable:
    name = ckpt.name if ckpt.name.startswith('checkpoint') else 'final'
    folder = out_root / name
    folder.mkdir(parents=True, exist_ok=True)

    pipe.load_lora_weights(str(ckpt))
    for key, prompt in PROMPTS.items():
        for seed in SEEDS:
            generator = torch.Generator('cuda').manual_seed(seed)
            image = pipe(
                prompt=prompt, negative_prompt=NEGATIVE,
                num_inference_steps=28, guidance_scale=6.0,
                width=1024, height=1024, generator=generator,
            ).images[0]
            image.save(folder / (key + '_' + str(seed) + '.png'))
    pipe.unload_lora_weights()
    print('เจนเสร็จ', name)
    gc.collect(); torch.cuda.empty_cache()

print('ทั้งหมดอยู่ที่', out_root)


## 8. ดูผลเทียบกัน


In [ ]:
import pathlib
from PIL import Image
import matplotlib.pyplot as plt

KEY = 'E'      # เปลี่ยนเป็น A-J เพื่อดู prompt อื่น
SEED = 1337

root = pathlib.Path('/content/samples')
if not root.exists():
    raise SystemExit('ยังไม่มีภาพตัวอย่าง ให้กลับไปรันเซลล์เจนภาพ (ข้อ 7) ให้ผ่านก่อน')

folders = sorted(root.iterdir(), key=lambda p: (p.name == 'final', p.name))
found = [(f.name, f / (KEY + '_' + str(SEED) + '.png')) for f in folders]
found = [(n, p) for n, p in found if p.exists()]

if not found:
    available = sorted({q.name.split('_')[0] for f in folders for q in f.glob('*.png')})
    raise SystemExit('ไม่พบภาพของ prompt ' + KEY + ' seed ' + str(SEED) +
                     ' · prompt ที่มี: ' + ', '.join(available))

fig, axes = plt.subplots(1, len(found), figsize=(4 * len(found), 4.6))
if len(found) == 1:
    axes = [axes]
for ax, (name, path) in zip(axes, found):
    ax.imshow(Image.open(path)); ax.set_title(name, fontsize=10); ax.axis('off')
plt.suptitle('prompt ' + KEY + ' · seed ' + str(SEED))
plt.tight_layout(); plt.show()


## 9. ดาวน์โหลด LoRA กับภาพตัวอย่าง

เอา `samples.zip` ไปให้คะแนนด้วย `evaluate.py` บนเครื่อง
และเอา `lora_weights.zip` ไปใช้เจนจริงต่อ


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/lora_weights', 'zip', '/content/lora_out')
shutil.make_archive('/content/samples', 'zip', '/content/samples')
files.download('/content/lora_weights.zip')
files.download('/content/samples.zip')


---

## ถ้า T4 ไม่ไหว ให้ใช้ SD 1.5 แทน

SD 1.5 เทรนเร็วกว่า SDXL ราว 5 เท่าและกินหน่วยความจำน้อยกว่ามาก
พื้นผิวสู้ SDXL ไม่ได้ แต่ถ้า Colab หลุดบ่อยก็คุ้มกว่า
เอาคำสั่งข้างล่างไปแทนเซลล์ที่ 6 (ลบ # ออกก่อน)


In [ ]:
# !accelerate launch --mixed_precision=fp16 \
#   /content/diffusers_src/examples/text_to_image/train_text_to_image_lora.py \
#   --pretrained_model_name_or_path='runwayml/stable-diffusion-v1-5' \
#   --train_data_dir='/content/dataset_aug/images' --caption_column='text' \
#   --resolution=512 --random_flip --train_batch_size=2 \
#   --gradient_accumulation_steps=2 --max_train_steps=2000 \
#   --checkpointing_steps=400 --learning_rate=1e-4 \
#   --lr_scheduler='cosine' --lr_warmup_steps=50 --rank=16 \
#   --seed=1337 --mixed_precision='fp16' --output_dir='/content/lora_out'
